# Laboratorio: Modelos Generativos Profundos — VAE, Flujos Normalizantes y Modelos Basados en Energía

**MDS221 Inteligencia Artificial — Magíster en Data Science, UCM** · Complemento de la clase 2.7 (*Modelos Generativos Profundos*).

En la clase vimos que todo modelo generativo intenta aproximar $p_{\text{data}}(x)$ y que el obstáculo común es la **normalización** $\int p_\theta(x)\,dx = 1$. Cada familia resuelve ese problema de manera distinta:

| Parte | Modelo | Estrategia ante la normalización | Datos |
|---|---|---|---|
| A | Autoencoder variacional (VAE) | Cota inferior (ELBO) | MNIST |
| B | Flujo normalizante (RealNVP) | Cambio de variables con jacobiano barato | *Two moons* 2D |
| C | Modelo basado en energía (EBM) | Densidad no normalizada + Langevin | *Two moons* 2D |
| D (opcional) | Denoising score matching | Aprender el *score* $\nabla_x \log p(x)$ | *Two moons* 2D |

Los laboratorios 2.7 (GAN) y 2.8 (autoencoders) ya cubren la GAN y un VAE denso; aquí construimos un **VAE convolucional** y lo evaluamos como modelo de densidad, e implementamos desde cero los otros dos.

**Objetivos de aprendizaje**
1. Implementar el ELBO, el truco de reparametrización y el KL gaussiano en forma cerrada.
2. Estimar $\log p_\theta(x)$ con muestreo de importancia y medir la brecha del ELBO.
3. Implementar capas de acoplamiento afín, verificar su inversa y su log-determinante.
4. Entrenar un EBM con divergencia contrastiva persistente y muestreo de Langevin.
5. Comparar las tres familias en calidad, velocidad de muestreo y acceso a la densidad.

> Se recomienda ejecutar en **Colab con GPU** (Entorno de ejecución → Cambiar tipo → T4). Las partes B, C y D corren en segundos en CPU.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from sklearn.datasets import make_moons

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', device)

torch.manual_seed(0)
np.random.seed(0)
plt.rcParams['figure.dpi'] = 100

---
# Parte A — Autoencoder Variacional Convolucional

## A.1 Modelo y función objetivo

Recordemos el modelo de variable latente de la clase:

$$z \sim \mathcal{N}(0, I_d), \qquad x \mid z \sim p_\theta(x \mid z) = \prod_{j} \text{Bernoulli}\big(x_j;\, f_\theta(z)_j\big)$$

y el codificador gaussiano $q_\phi(z\mid x) = \mathcal{N}\big(\mu_\phi(x), \mathrm{diag}(\sigma^2_\phi(x))\big)$. Maximizamos el ELBO, o equivalentemente minimizamos

$$\mathcal{L}_{\text{VAE}}(x) = \underbrace{-\,\mathbb{E}_{q_\phi(z\mid x)}\big[\log p_\theta(x\mid z)\big]}_{\text{entropía cruzada binaria}} + \beta\,\underbrace{\frac{1}{2}\sum_{j=1}^d \big(\mu_j^2 + \sigma_j^2 - \log\sigma_j^2 - 1\big)}_{\mathrm{KL}(q_\phi(z|x)\,\|\,p(z))}$$

con $\beta = 1$ para el VAE estándar. El gradiente respecto a $\phi$ pasa por la **reparametrización** $z = \mu + \sigma \odot \epsilon$, $\epsilon \sim \mathcal{N}(0, I)$.

## A.2 Datos

Usamos MNIST con intensidades en $[0,1]$ (la verosimilitud Bernoulli acepta valores continuos en ese rango; esto es una aproximación habitual).

In [ ]:
transform = transforms.ToTensor()
train_ds = torchvision.datasets.MNIST('data', train=True, download=True, transform=transform)
test_ds  = torchvision.datasets.MNIST('data', train=False, download=True, transform=transform)

batch_size = 128
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

x_ej, y_ej = next(iter(test_loader))
fig, axs = plt.subplots(1, 10, figsize=(10, 1.2))
for i, ax in enumerate(axs):
    ax.imshow(x_ej[i, 0], cmap='gray'); ax.set_title(int(y_ej[i])); ax.axis('off')
plt.show()

## A.3 Arquitectura

* **Codificador** $g_\phi$: dos convoluciones con *stride* 2 ($28 \to 14 \to 7$) y dos cabezas lineales que entregan $\mu$ y $\log\sigma^2$ (predecir el logaritmo garantiza $\sigma^2 > 0$ y es numéricamente estable).
* **Decodificador** $f_\theta$: la imagen especular con convoluciones transpuestas. Devuelve **logits**; la sigmoide se aplica dentro de la pérdida (`binary_cross_entropy_with_logits`), lo que es más estable que `Sigmoid` + `BCE`.

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 4, stride=2, padding=1), nn.ReLU(),   # 32 x 14 x 14
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(),  # 64 x 7 x 7
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 64 * 7 * 7), nn.ReLU(),
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),  # 32 x 14 x 14
            nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1),             # 1 x 28 x 28 (logits)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparametrize(self, mu, logvar):
        # z = mu + sigma * eps, con eps ~ N(0, I)
        eps = torch.randn_like(mu)
        return mu + torch.exp(0.5 * logvar) * eps

    def decode(self, z):
        return self.decoder(z)   # logits

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparametrize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(logits, x, mu, logvar, beta=1.0):
    # Promedios por imagen: -ELBO = reconstrucción + beta * KL
    rec = F.binary_cross_entropy_with_logits(logits, x, reduction='sum') / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return rec + beta * kl, rec, kl


vae = ConvVAE(latent_dim=2).to(device)
print(f'Parámetros: {sum(p.numel() for p in vae.parameters()):,}')
logits, mu, logvar = vae(x_ej.to(device))
print('logits:', tuple(logits.shape), ' mu:', tuple(mu.shape))

> **Pregunta A.1** — Demuestre a partir de la fórmula de la clase que $\mathrm{KL}\big(\mathcal{N}(\mu,\sigma^2)\,\|\,\mathcal{N}(0,1)\big) = \tfrac{1}{2}(\mu^2 + \sigma^2 - \log\sigma^2 - 1)$ y verifique que la línea `kl = ...` implementa exactamente esa expresión sumada sobre las $d$ dimensiones.

## A.4 Entrenamiento

Registramos por separado los dos términos del ELBO para ver cómo compiten.

In [ ]:
def train_vae(model, epochs=10, beta=1.0, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {'loss': [], 'rec': [], 'kl': [], 'test': []}
    for ep in range(epochs):
        model.train()
        tot = rec_t = kl_t = 0.0; n = 0
        for x, _ in train_loader:
            x = x.to(device)
            logits, mu, logvar = model(x)
            loss, rec, kl = vae_loss(logits, x, mu, logvar, beta)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * x.size(0); rec_t += rec.item() * x.size(0)
            kl_t += kl.item() * x.size(0); n += x.size(0)
        hist['loss'].append(tot / n); hist['rec'].append(rec_t / n); hist['kl'].append(kl_t / n)
        hist['test'].append(evaluate_elbo(model, beta))
        print(f'época {ep+1:2d} | -ELBO train {tot/n:7.2f} | rec {rec_t/n:7.2f} | KL {kl_t/n:5.2f} | -ELBO test {hist["test"][-1]:7.2f}')
    return hist


@torch.no_grad()
def evaluate_elbo(model, beta=1.0):
    model.eval()
    tot = 0.0; n = 0
    for x, _ in test_loader:
        x = x.to(device)
        logits, mu, logvar = model(x)
        loss, _, _ = vae_loss(logits, x, mu, logvar, beta)
        tot += loss.item() * x.size(0); n += x.size(0)
    return tot / n


hist2 = train_vae(vae, epochs=10)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
axs[0].plot(hist2['loss'], label='-ELBO (train)'); axs[0].plot(hist2['test'], label='-ELBO (test)')
axs[0].set_xlabel('época'); axs[0].set_ylabel('nats por imagen'); axs[0].legend()
axs[1].plot(hist2['rec'], label='reconstrucción'); axs[1].plot(hist2['kl'], label='KL')
axs[1].set_xlabel('época'); axs[1].set_yscale('log'); axs[1].legend()
plt.tight_layout(); plt.show()

## A.5 Reconstrucciones y espacio latente

Con $d = 2$ podemos **ver** el espacio latente: graficamos $\mu_\phi(x)$ para las imágenes de prueba coloreadas por dígito.

In [ ]:
@torch.no_grad()
def show_reconstructions(model, n=10):
    model.eval()
    x = x_ej[:n].to(device)
    xh = torch.sigmoid(model(x)[0]).cpu()
    fig, axs = plt.subplots(2, n, figsize=(n, 2.2))
    for i in range(n):
        axs[0, i].imshow(x_ej[i, 0], cmap='gray'); axs[1, i].imshow(xh[i, 0], cmap='gray')
        axs[0, i].axis('off'); axs[1, i].axis('off')
    axs[0, 0].set_title('original', loc='left', fontsize=9); axs[1, 0].set_title('reconstrucción', loc='left', fontsize=9)
    plt.show()

@torch.no_grad()
def latent_codes(model):
    model.eval()
    Z, Y = [], []
    for x, y in test_loader:
        mu, _ = model.encode(x.to(device)); Z.append(mu.cpu()); Y.append(y)
    return torch.cat(Z).numpy(), torch.cat(Y).numpy()

show_reconstructions(vae)
Z, Y = latent_codes(vae)
plt.figure(figsize=(6, 5))
sc = plt.scatter(Z[:, 0], Z[:, 1], c=Y, cmap='tab10', s=2, alpha=0.7)
plt.colorbar(sc, ticks=range(10)); plt.xlabel('$z_1$'); plt.ylabel('$z_2$')
plt.title('Medias del codificador $\\mu_\\phi(x)$ (test)'); plt.axis('equal'); plt.show()

### Variedad decodificada

Recorremos una grilla de cuantiles de $\mathcal{N}(0,1)$ en cada eje (con $\Phi^{-1}$, para que la grilla cubra la masa del prior de manera uniforme) y decodificamos cada punto.

In [ ]:
from scipy.stats import norm

@torch.no_grad()
def plot_manifold(model, n=15):
    model.eval()
    q = norm.ppf(np.linspace(0.03, 0.97, n))
    canvas = np.zeros((28 * n, 28 * n))
    for i, yi in enumerate(q[::-1]):
        z = torch.tensor([[xi, yi] for xi in q], dtype=torch.float32, device=device)
        imgs = torch.sigmoid(model.decode(z)).cpu().numpy()
        for j in range(n):
            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = imgs[j, 0]
    plt.figure(figsize=(7, 7)); plt.imshow(canvas, cmap='gray'); plt.axis('off')
    plt.title('Decodificación de una grilla en el espacio latente'); plt.show()

plot_manifold(vae)

> **Pregunta A.2** — Compare el gráfico de dispersión con el de un autoencoder determinista del laboratorio 2.8. ¿Qué efecto visible tiene el término KL sobre la forma y la escala de la nube de códigos? Relaciónelo con la diapositiva *"El Espacio Latente de un Autoencoder"*.

## A.6 Un VAE con más capacidad: muestreo e interpolación

Con $d = 2$ las muestras son borrosas porque el cuello de botella es muy estrecho. Entrenamos un segundo modelo con $d = 16$ para generar e interpolar.

In [ ]:
vae16 = ConvVAE(latent_dim=16).to(device)
hist16 = train_vae(vae16, epochs=15)
show_reconstructions(vae16)

In [ ]:
@torch.no_grad()
def sample_grid(model, n=64):
    model.eval()
    z = torch.randn(n, model.latent_dim, device=device)
    x = torch.sigmoid(model.decode(z)).cpu()
    grid = torchvision.utils.make_grid(x, nrow=16, pad_value=1)
    plt.figure(figsize=(12, 3.2)); plt.imshow(grid.permute(1, 2, 0)); plt.axis('off')
    plt.title('Muestras: $z \\sim \\mathcal{N}(0, I)$, $\\hat x = f_\\theta(z)$'); plt.show()

@torch.no_grad()
def interpolate(model, xa, xb, steps=10):
    model.eval()
    za, _ = model.encode(xa[None].to(device)); zb, _ = model.encode(xb[None].to(device))
    t = torch.linspace(0, 1, steps, device=device)[:, None]
    imgs = torch.sigmoid(model.decode((1 - t) * za + t * zb)).cpu()
    fig, axs = plt.subplots(1, steps, figsize=(steps, 1.2))
    for i, ax in enumerate(axs):
        ax.imshow(imgs[i, 0], cmap='gray'); ax.axis('off')
    plt.show()

sample_grid(vae16)
interpolate(vae16, test_ds[3][0], test_ds[2][0])   # un 0 hacia un 1
interpolate(vae16, test_ds[0][0], test_ds[1][0])   # un 7 hacia un 2

## A.7 ¿Cuánto vale realmente $\log p_\theta(x)$? La brecha del ELBO

El ELBO es sólo una **cota inferior**. Con muestreo de importancia usando $q_\phi$ como propuesta obtenemos una cota mucho más ajustada (IWAE, Burda et al. 2016):

$$\log p_\theta(x) \;\ge\; \log \frac{1}{K}\sum_{k=1}^{K} \frac{p_\theta(x \mid z_k)\, p(z_k)}{q_\phi(z_k \mid x)}, \qquad z_k \sim q_\phi(z\mid x),$$

que coincide con el ELBO para $K=1$ y converge a $\log p_\theta(x)$ cuando $K \to \infty$. La diferencia aproxima el término $\mathrm{KL}\big(q_\phi(z|x)\,\|\,p_\theta(z|x)\big)$ de la clase.

In [ ]:
@torch.no_grad()
def log_px_importance(model, x, K=200):
    # Estimador IWAE de log p(x) para un minibatch x; devuelve un tensor (B,)
    model.eval()
    mu, logvar = model.encode(x)
    std = torch.exp(0.5 * logvar)
    B, d = mu.shape
    eps = torch.randn(K, B, d, device=x.device)
    z = mu + std * eps                                           # (K, B, d)
    logits = model.decode(z.reshape(K * B, d)).reshape(K, B, -1)
    log_px_z = -F.binary_cross_entropy_with_logits(
        logits, x.reshape(1, B, -1).expand(K, B, -1), reduction='none').sum(-1)
    log_pz = (-0.5 * z.pow(2) - 0.5 * math.log(2 * math.pi)).sum(-1)
    log_qz = (-0.5 * eps.pow(2) - 0.5 * math.log(2 * math.pi) - torch.log(std)).sum(-1)
    return torch.logsumexp(log_px_z + log_pz - log_qz, dim=0) - math.log(K)

x = x_ej[:64].to(device)
for name, m in [('d=2', vae), ('d=16', vae16)]:
    logits, mu, logvar = m(x)
    elbo = -vae_loss(logits, x, mu, logvar)[0].item()
    for K in [1, 10, 100, 500]:
        print(f'{name:5s} K={K:4d}  cota de log p(x) = {log_px_importance(m, x, K).mean().item():8.2f} nats')
    print(f'{name:5s} ELBO (1 muestra)         = {elbo:8.2f} nats\n')

> **Pregunta A.3** — ¿Cuántos nats separan el ELBO de la estimación con $K = 500$? ¿Qué modelo tiene la brecha más grande y por qué? Exprese el mejor $\log p(x)$ en **bits por dimensión** ($-\log_2 p(x) / 784$).

## A.8 Ejercicios de la Parte A

1. **$\beta$-VAE.** Entrene `ConvVAE(latent_dim=2)` con $\beta = 0.5$ y $\beta = 4$ (`train_vae(..., beta=...)`). Compare el diagrama de dispersión latente, la variedad decodificada y la nitidez de las reconstrucciones. ¿Qué compromiso observa?
2. **Colapso del posterior.** Entrene con $\beta = 20$. Calcule el KL promedio por dimensión latente sobre el conjunto de prueba (`0.5*(mu**2 + logvar.exp() - logvar - 1)`). ¿Cuántas dimensiones quedan "activas" (KL $> 0.01$)?
3. **Detección de anomalías.** Cargue Fashion-MNIST (`torchvision.datasets.FashionMNIST`) y compare el histograma de $\log p_\theta(x)$ (IWAE, $K=100$) entre MNIST de prueba y Fashion-MNIST usando `vae16`. ¿Separa bien ambas distribuciones? Calcule el AUROC con `sklearn.metrics.roc_auc_score`.

---
# Parte B — Flujo Normalizante RealNVP

## B.1 Datos: *two moons*

Trabajamos en 2D para poder visualizar la densidad completa. Estandarizamos los datos.

In [ ]:
def moons(n, noise=0.05):
    X, _ = make_moons(n_samples=n, noise=noise)
    X = (X - np.array([0.5, 0.25])) / np.array([0.87, 0.5])     # estandarización aproximada
    return torch.tensor(X, dtype=torch.float32)

X_moons = moons(4000)
plt.figure(figsize=(4.5, 4)); plt.scatter(X_moons[:, 0], X_moons[:, 1], s=2)
plt.title('Datos: two moons'); plt.axis('equal'); plt.show()

## B.2 Capa de acoplamiento afín

Con una máscara binaria $m$ (qué coordenadas pasan intactas), la capa transforma, en la dirección **datos $\to$ latente**:

$$z = m \odot x + (1 - m) \odot \Big(\big(x - t(m\odot x)\big) \odot e^{-s(m \odot x)}\Big), \qquad \log\left|\det \frac{\partial z}{\partial x}\right| = -\sum_j (1 - m_j)\, s_j(m\odot x)$$

y su inversa (**latente $\to$ datos**, para muestrear):

$$x = m \odot z + (1 - m) \odot \Big(z \odot e^{s(m\odot z)} + t(m\odot z)\Big).$$

Las redes $s$ y $t$ son MLP arbitrarios: **nunca necesitan invertirse**. Acotamos $s$ con `tanh` para estabilidad.

In [ ]:
class AffineCoupling(nn.Module):
    def __init__(self, mask, hidden=128):
        super().__init__()
        self.register_buffer('mask', mask)
        D = mask.numel()
        self.net = nn.Sequential(
            nn.Linear(D, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2 * D))
        self.scale = nn.Parameter(torch.zeros(D))   # escala aprendible para s, parte en 0 (identidad)

    def _st(self, x_masked):
        s, t = self.net(x_masked).chunk(2, dim=-1)
        s = self.scale * torch.tanh(s)
        return s * (1 - self.mask), t * (1 - self.mask)

    def forward(self, x):          # datos -> latente; devuelve z y log|det dz/dx|
        s, t = self._st(x * self.mask)
        z = x * self.mask + (1 - self.mask) * (x - t) * torch.exp(-s)
        return z, -s.sum(-1)

    def inverse(self, z):          # latente -> datos
        s, t = self._st(z * self.mask)
        return z * self.mask + (1 - self.mask) * (z * torch.exp(s) + t)


class RealNVP(nn.Module):
    def __init__(self, D=2, n_layers=8, hidden=128):
        super().__init__()
        masks = [torch.tensor([(j + i) % 2 for j in range(D)], dtype=torch.float32) for i in range(n_layers)]
        self.layers = nn.ModuleList([AffineCoupling(m, hidden) for m in masks])
        self.D = D

    def forward(self, x):          # x -> z, suma de log-determinantes
        logdet = torch.zeros(x.size(0), device=x.device)
        for layer in self.layers:
            x, ld = layer(x); logdet = logdet + ld
        return x, logdet

    def inverse(self, z, return_path=False):
        path = [z]
        for layer in reversed(self.layers):
            z = layer.inverse(z); path.append(z)
        return (z, path) if return_path else z

    def log_prob(self, x):         # cambio de variables
        z, logdet = self.forward(x)
        log_pz = -0.5 * (z ** 2).sum(-1) - 0.5 * self.D * math.log(2 * math.pi)
        return log_pz + logdet

    def sample(self, n):
        return self.inverse(torch.randn(n, self.D, device=self.layers[0].mask.device))

## B.3 Verificaciones numéricas (antes de entrenar)

Un flujo sólo es correcto si (1) la inversa es exacta y (2) el log-determinante coincide con el del jacobiano verdadero. Lo comprobamos con `torch.autograd.functional.jacobian` en un flujo con pesos aleatorios.

In [ ]:
flow_test = RealNVP()
for layer in flow_test.layers:                         # pesos no triviales para la prueba
    nn.init.normal_(layer.scale, std=0.5)

x = torch.randn(5, 2)
z, logdet = flow_test(x)
print('Error de reconstrucción |f^{-1}(f(x)) - x| =', (flow_test.inverse(z) - x).abs().max().item())

for i in range(3):
    J = torch.autograd.functional.jacobian(lambda v: flow_test(v[None])[0][0], x[i])
    print(f'log|det J| autograd = {torch.linalg.slogdet(J).logabsdet.item():+.6f}   |   flujo = {logdet[i].item():+.6f}')

## B.4 Entrenamiento por máxima verosimilitud exacta

$$\theta^* = \arg\min_\theta\; -\frac{1}{N}\sum_i \Big[\log p_Z\big(f_\theta^{-1}(x^{(i)})\big) + \log\Big|\det \tfrac{\partial f_\theta^{-1}}{\partial x}\Big|\Big]$$

In [ ]:
flow = RealNVP(n_layers=8).to(device)
opt = torch.optim.Adam(flow.parameters(), lr=1e-3)
losses = []
for it in range(3000):
    x = moons(512).to(device)
    loss = -flow.log_prob(x).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if (it + 1) % 500 == 0:
        print(f'iter {it+1:5d} | NLL = {np.mean(losses[-100:]):.3f} nats')

plt.figure(figsize=(5, 3)); plt.plot(losses); plt.xlabel('iteración'); plt.ylabel('NLL'); plt.show()

In [ ]:
def density_grid(logp_fn, lim=2.5, n=200):
    g = torch.linspace(-lim, lim, n)
    xx, yy = torch.meshgrid(g, g, indexing='xy')
    pts = torch.stack([xx.flatten(), yy.flatten()], 1).to(device)
    with torch.no_grad():
        lp = logp_fn(pts).reshape(n, n).cpu()
    return xx, yy, lp

xx, yy, lp = density_grid(flow.log_prob)
with torch.no_grad():
    samples = flow.sample(2000).cpu()

fig, axs = plt.subplots(1, 3, figsize=(13, 4))
axs[0].scatter(X_moons[:, 0], X_moons[:, 1], s=2); axs[0].set_title('Datos')
axs[1].pcolormesh(xx, yy, lp.exp(), shading='auto', cmap='magma'); axs[1].set_title('Densidad $p_\\theta(x)$ (exacta)')
axs[2].scatter(samples[:, 0], samples[:, 1], s=2, c='C3'); axs[2].set_title('Muestras $x = f_\\theta(z)$')
for ax in axs: ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5); ax.set_aspect('equal')
plt.show()

with torch.no_grad():
    dx = 5.0 / 199
    print('Integral numérica de p(x) en la grilla ≈', (lp.exp().sum() * dx * dx).item())

### Visualizar la composición capa por capa

Las muestras de una gaussiana se van deformando hasta convertirse en las dos lunas. Coloreamos cada punto por su ángulo inicial en $z$ para seguir cómo se mueve.

In [ ]:
with torch.no_grad():
    z0 = torch.randn(3000, 2, device=device)
    _, path = flow.inverse(z0, return_path=True)
color = torch.atan2(z0[:, 1], z0[:, 0]).cpu()
idx = [0, 2, 4, 6, 8]
fig, axs = plt.subplots(1, len(idx), figsize=(3 * len(idx), 3))
for ax, k in zip(axs, idx):
    p = path[k].cpu()
    ax.scatter(p[:, 0], p[:, 1], s=1, c=color, cmap='hsv')
    ax.set_title('$z$ (base)' if k == 0 else f'tras {k} capas'); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect('equal')
plt.show()

> **Pregunta B.1** — En la densidad estimada suele aparecer un "puente" tenue de probabilidad entre las dos lunas. Explique por qué un flujo (un homeomorfismo de $\mathbb{R}^2$) no puede separar completamente una gaussiana en dos componentes desconectadas (diapositiva *"Entrenamiento y Balance de los Flujos"*).

### Ejercicios de la Parte B
1. Entrene con `n_layers` $\in \{2, 4, 16\}$ y compare la NLL final y la densidad. ¿Cuántos parámetros tiene cada modelo?
2. Ajuste una mezcla de gaussianas (`sklearn.mixture.GaussianMixture`, 2 a 10 componentes) a los mismos datos y compare su log-verosimilitud promedio con la del flujo.
3. Cronometre `flow.sample(10_000)` y `flow.log_prob(x)` para 10 000 puntos. ¿Cuál es más caro? ¿Cambiaría la respuesta para un flujo autorregresivo (MAF)?

---
# Parte C — Modelo Basado en Energía

## C.1 Modelo

Una red escalar $E_\theta : \mathbb{R}^2 \to \mathbb{R}$ define $p_\theta(x) = e^{-E_\theta(x)} / Z_\theta$. No calculamos $Z_\theta$ en ningún momento.

**Entrenamiento (divergencia contrastiva persistente):**
$$\mathcal{L}(\theta) = \frac{1}{B}\sum_i E_\theta(x^{(i)}) - \frac{1}{B}\sum_i E_\theta\big(\mathrm{sg}[x'^{(i)}]\big) + \alpha\,\frac{1}{B}\sum_i \Big(E_\theta(x^{(i)})^2 + E_\theta(x'^{(i)})^2\Big)$$

**Muestras negativas** $x'$ con dinámica de Langevin, partiendo de un *buffer* de reinicio (95 % cadenas persistentes, 5 % ruido uniforme nuevo):
$$x_{k+1} = x_k - \frac{\eta}{2}\nabla_x E_\theta(x_k) + \sqrt{\eta}\,\epsilon_k$$

In [ ]:
class EnergyNet(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)


def langevin(energy, x, steps=60, step_size=0.01, noise=True):
    # Muestreo de Langevin no ajustado (ULA). El score -dE/dx no depende de Z.
    x = x.clone().detach()
    for _ in range(steps):
        x.requires_grad_(True)
        grad = torch.autograd.grad(energy(x).sum(), x)[0]
        x = x.detach() - 0.5 * step_size * grad
        if noise:
            x = x + math.sqrt(step_size) * torch.randn_like(x)
    return x.detach()

In [ ]:
ebm = EnergyNet().to(device)
opt = torch.optim.Adam(ebm.parameters(), lr=1e-3)
buffer = (torch.rand(10_000, 2, device=device) * 6 - 3)     # buffer de reinicio: U[-3, 3]^2
alpha, B = 0.1, 256
hist_ebm = {'pos': [], 'neg': []}

for it in range(4000):
    x_pos = moons(B).to(device)
    # 1) inicializar cadenas: 95 % desde el buffer, 5 % ruido nuevo
    idx = torch.randint(0, buffer.size(0), (B,), device=device)
    x0 = buffer[idx]
    fresh = torch.rand(B, device=device) < 0.05
    x0[fresh] = torch.rand(int(fresh.sum()), 2, device=device) * 6 - 3
    # 2) fase negativa: Langevin desde x0
    x_neg = langevin(ebm, x0, steps=30, step_size=0.01)
    buffer[idx] = x_neg
    # 3) actualización contrastiva
    e_pos, e_neg = ebm(x_pos), ebm(x_neg)
    loss = e_pos.mean() - e_neg.mean() + alpha * (e_pos.pow(2).mean() + e_neg.pow(2).mean())
    opt.zero_grad(); loss.backward(); opt.step()
    hist_ebm['pos'].append(e_pos.mean().item()); hist_ebm['neg'].append(e_neg.mean().item())
    if (it + 1) % 500 == 0:
        print(f'iter {it+1:5d} | E(datos) = {e_pos.mean().item():+.3f} | E(negativos) = {e_neg.mean().item():+.3f}')

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(hist_ebm['pos'], label='$E_\\theta$(datos)  fase positiva', alpha=0.8)
plt.plot(hist_ebm['neg'], label='$E_\\theta$(negativos)  fase negativa', alpha=0.8)
plt.xlabel('iteración'); plt.legend(); plt.show()

La densidad **no normalizada** $e^{-E_\theta(x)}$ se puede graficar, y en 2D podemos normalizarla numéricamente en la grilla (algo imposible en alta dimensión).

In [ ]:
xx, yy, negE = density_grid(lambda p: -ebm(p))
dens = (negE - negE.max()).exp(); dens = dens / (dens.sum() * (5.0 / 199) ** 2)

x_init = torch.rand(2000, 2, device=device) * 6 - 3
samples_ebm = langevin(ebm, x_init, steps=500, step_size=0.01).cpu()

fig, axs = plt.subplots(1, 3, figsize=(13, 4))
c = axs[0].contourf(xx, yy, -negE, levels=30, cmap='viridis'); plt.colorbar(c, ax=axs[0]); axs[0].set_title('Energía $E_\\theta(x)$')
axs[1].pcolormesh(xx, yy, dens, shading='auto', cmap='magma'); axs[1].set_title('$e^{-E_\\theta}/Z$ (normalizada en la grilla)')
axs[2].scatter(samples_ebm[:, 0], samples_ebm[:, 1], s=2, c='C3'); axs[2].set_title('500 pasos de Langevin desde ruido')
for ax in axs: ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5); ax.set_aspect('equal')
plt.show()

### Trayectorias de Langevin

Seguimos unas pocas cadenas para ver el descenso por la energía más el ruido (diapositiva *"Muestreo con Dinámica de Langevin"*).

In [ ]:
x = torch.rand(8, 2, device=device) * 5 - 2.5
traj = [x.cpu()]
for _ in range(40):
    x = langevin(ebm, x, steps=5, step_size=0.01); traj.append(x.cpu())
traj = torch.stack(traj)          # (pasos, cadenas, 2)

plt.figure(figsize=(5, 5))
plt.contourf(xx, yy, -negE, levels=30, cmap='viridis', alpha=0.8)
for k in range(traj.size(1)):
    plt.plot(traj[:, k, 0], traj[:, k, 1], '-', color='white', lw=0.8)
    plt.plot(traj[0, k, 0], traj[0, k, 1], 'o', color='C3', ms=4)
plt.xlim(-2.5, 2.5); plt.ylim(-2.5, 2.5); plt.gca().set_aspect('equal')
plt.title('Cadenas de Langevin (rojo: inicio)'); plt.show()

> **Pregunta C.1** — Repita el muestreo final con `noise=False` (descenso de gradiente puro sobre $E_\theta$). ¿Qué ocurre con la diversidad de las muestras? ¿Por qué el término $\sqrt{\eta}\,\epsilon$ es indispensable para muestrear de $p_\theta$ y no sólo encontrar sus modos?

> **Pregunta C.2** — Compare el panel central (densidad $e^{-E_\theta}/Z$) con el derecho (muestras de Langevin): las muestras suelen verse **más dispersas** que la densidad, y algunas quedan fuera de las lunas. Dé dos razones: (i) el sesgo de Langevin no ajustado con paso $\eta$ finito y (ii) que en las zonas planas de alta energía $\nabla_x E_\theta \approx 0$, por lo que las cadenas sólo difunden. ¿Cómo cambia el resultado con `step_size=0.002` y más pasos?

### Ejercicios de la Parte C
1. Elimine la regularización ($\alpha = 0$). ¿Qué sucede con la escala de las energías durante el entrenamiento?
2. Reemplace el buffer persistente por cadenas cortas que parten siempre de ruido (`x0 = rand`). Compare la calidad del paisaje de energía con el mismo número de pasos.
3. Mida el tiempo para generar 2000 muestras con el EBM (500 pasos de Langevin) y con el flujo (una pasada). Relaciónelo con la tabla comparativa de la clase.

---
# Parte D (opcional) — Del EBM a la Difusión: Denoising Score Matching

En lugar de aprender $E_\theta$ y derivarla, aprendemos directamente el **score** $s_\theta(x, \sigma) \approx \nabla_x \log p_\sigma(x)$ de los datos perturbados con ruido gaussiano de nivel $\sigma$. El objetivo de *denoising score matching* no requiere $Z_\theta$ ni muestreo durante el entrenamiento:

$$\mathcal{L}(\theta) = \mathbb{E}_{\sigma}\,\mathbb{E}_{x,\epsilon}\Big[\sigma^2\,\big\|s_\theta(x + \sigma\epsilon, \sigma) + \epsilon/\sigma\big\|^2\Big]$$

Para muestrear se usa **Langevin recocido**: Langevin con $\sigma$ grande y luego cada vez más pequeño. Esta es la idea base de los modelos de difusión (Song & Ermon, 2019).

In [ ]:
sigmas = torch.exp(torch.linspace(math.log(1.0), math.log(0.01), 10)).to(device)

class ScoreNet(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2))
    def forward(self, x, sigma):
        # se condiciona en log(sigma) y se divide por sigma (parametrización estándar)
        return self.net(torch.cat([x, sigma.log()[:, None]], 1)) / sigma[:, None]

score = ScoreNet().to(device)
opt = torch.optim.Adam(score.parameters(), lr=1e-3)
for it in range(5000):
    x = moons(512).to(device)
    sig = sigmas[torch.randint(0, len(sigmas), (x.size(0),), device=device)]
    eps = torch.randn_like(x)
    loss = (sig[:, None] ** 2 * (score(x + sig[:, None] * eps, sig) + eps / sig[:, None]) ** 2).sum(-1).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if (it + 1) % 1000 == 0:
        print(f'iter {it+1:5d} | DSM = {loss.item():.4f}')

@torch.no_grad()
def annealed_langevin(n=2000, steps=100, eps=2e-5):
    x = torch.rand(n, 2, device=device) * 6 - 3
    for s in sigmas:
        a = eps * (s / sigmas[-1]) ** 2                      # paso proporcional a sigma^2
        sv = s.expand(n)
        for _ in range(steps):
            x = x + 0.5 * a * score(x, sv) + a.sqrt() * torch.randn_like(x)
    return x.cpu()

samples_dsm = annealed_langevin()
plt.figure(figsize=(4.5, 4)); plt.scatter(samples_dsm[:, 0], samples_dsm[:, 1], s=2, c='C2')
plt.xlim(-2.5, 2.5); plt.ylim(-2.5, 2.5); plt.gca().set_aspect('equal')
plt.title('Langevin recocido con score aprendido'); plt.show()

---
# Comparación y Entregable

Ejecute la siguiente celda para comparar los modelos 2D en una misma figura.

In [ ]:
panels = [('Datos', X_moons), ('RealNVP', samples), ('EBM + Langevin', samples_ebm)]
if 'samples_dsm' in globals():
    panels.append(('Score + Langevin recocido', samples_dsm))
fig, axs = plt.subplots(1, len(panels), figsize=(3.6 * len(panels), 3.6))
for ax, (t, s) in zip(axs, panels):
    ax.scatter(s[:, 0], s[:, 1], s=1.5); ax.set_title(t)
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5); ax.set_aspect('equal')
plt.show()

## Entregable

Entregue este notebook ejecutado, con las respuestas a las preguntas **A.1–A.3, B.1, C.1 y C.2** en celdas de texto y al menos **un ejercicio de cada parte (A, B, C)**. Complete además la tabla con sus propias mediciones:

| | VAE ($d=16$) | RealNVP | EBM |
|---|---|---|---|
| Acceso a la densidad (exacta / cota / no normalizada) | | | |
| Valor de log-verosimilitud medido (indique cuál) | | | |
| Tiempo para generar 2000 muestras | | | |
| Calidad visual (1–5) y comentario | | | |

Cierre con un párrafo (máx. 10 líneas): **¿qué familia usaría para (a) detectar anomalías en imágenes médicas, (b) generar datos sintéticos para entrenar un clasificador y (c) estimar una densidad de baja dimensión con verosimilitud exacta?** Justifique con lo observado.

## Referencias
- D. P. Kingma & M. Welling, *Auto-Encoding Variational Bayes*, ICLR 2014.
- Y. Burda, R. Grosse & R. Salakhutdinov, *Importance Weighted Autoencoders*, ICLR 2016.
- L. Dinh, J. Sohl-Dickstein & S. Bengio, *Density Estimation using Real NVP*, ICLR 2017.
- Y. Du & I. Mordatch, *Implicit Generation and Modeling with Energy-Based Models*, NeurIPS 2019.
- Y. Song & S. Ermon, *Generative Modeling by Estimating Gradients of the Data Distribution*, NeurIPS 2019.
- S. J. D. Prince, *Understanding Deep Learning*, MIT Press, 2023, caps. 16–18.